# Phase 2 - Feature Engineering & Preprocessing

## Objective
Transform raw customer data into model-ready features using:
- Feature engineering (business-driven)
- Preprocessing pipeline (scaling + encoding)
- SMOTE for class imbalance

This phase ensures reproducibility and prevents data leakage.


In [ ]:
import sys
import warnings
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

warnings.filterwarnings(
    "ignore",
    message="Could not infer format, so each element will be parsed individually.*",
    category=UserWarning,
)

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from src.features.build_features import build_features
from src.features.preprocessing_pipeline import create_pipeline
from src.data.make_dataset import clean_data, load_data, pick_raw_data_path, save_data, split_data

from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid")


## Step 3 - Load Data


In [ ]:
train_path = Path("../data/processed/train.csv")
if not train_path.exists():
    raw_path = pick_raw_data_path()
    raw_df = load_data(raw_path)
    raw_df = clean_data(raw_df)
    train_df, test_df = split_data(raw_df)
    save_data(train_df, test_df)

train = pd.read_csv(train_path)
train.head()


## Step 4 - Separate Target


In [ ]:
X = train.drop("Churn", axis=1)
y = train["Churn"].map({"Yes": 1, "No": 0})


## Step 5 - BEFORE Feature Engineering
Shows baseline relationship.


In [ ]:
sns.boxplot(x=y, y=train["MonthlyCharges"], hue=y, legend=False)
plt.title("Monthly Charges vs Churn (Before)")
plt.show()


## Step 6 - Apply Feature Engineering


In [ ]:
# If raw data already contains engineered columns, remove and rebuild for reproducibility.
engineered_cols = ["charges_per_month", "tenure_group", "HasMultiServices", "IsSeniorWithPartner", "ChargeToIncomeProxy"]
X_base = X.drop(columns=[c for c in engineered_cols if c in X.columns], errors="ignore")
X_fe = build_features(X_base)
X_fe.head()


## Step 7 - Inspect New Features


In [ ]:
new_cols = set(X_fe.columns) - set(X_base.columns)
print("New Features:", new_cols)


Expected key features: `charges_per_month`, `tenure_group`, `HasMultiServices`, `IsSeniorWithPartner`, `ChargeToIncomeProxy`.


## Step 8 - AFTER Feature Engineering


In [ ]:
sns.boxplot(x=y, y=X_fe["charges_per_month"], hue=y, legend=False)
plt.title("Charges per Month vs Churn (After)")
plt.show()
print("Insight: Engineered features reveal clearer separation between churn and non-churn customers.")


## Step 9 - Identify Feature Types


In [ ]:
numeric_features = X_fe.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_fe.select_dtypes(include=["object", "category"]).columns

print("Numeric:", list(numeric_features))
print("Categorical:", list(categorical_features))


## Step 10 - Build Preprocessing Pipeline


In [ ]:
pipeline = create_pipeline(numeric_features, categorical_features)


## Step 11 - Apply Pipeline
Scaling applied to numeric features, encoding applied to categorical features, and leakage is prevented by fitting only on train.


In [ ]:
# WoE encoding is supervised, so y is passed only for training fit.
X_processed = pipeline.fit_transform(X_fe, y)


## Step 12 - Shape Check


In [ ]:
print("Before:", X_fe.shape)
print("After:", X_processed.shape)


## Step 13 - Apply SMOTE


In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)


## Step 14 - Check Class Balance


In [ ]:
sns.countplot(x=y_resampled, hue=y_resampled, legend=False)
plt.title("Target Distribution After SMOTE")
plt.show()
print("Insight: Dataset is now balanced (50:50), improving model learning.")


## Step 15 - Final Dataset Check


In [ ]:
print("Final X shape:", X_resampled.shape)
print("Final y shape:", y_resampled.shape)


## Step 16 - Save Processed Data


In [ ]:
np.save("../data/processed/X_train.npy", X_resampled)
np.save("../data/processed/y_train.npy", y_resampled)


## Key Takeaways

1. Feature engineering improved signal clarity between churn and non-churn customers.
2. Derived features like charges_per_month and tenure_group capture customer lifecycle better.
3. WoE encoding provides more meaningful representation than one-hot encoding.
4. SMOTE successfully balanced the dataset, addressing class imbalance.

### Final Result
A fully processed, balanced dataset ready for model training.
